# Task 1 - Inductive Biases and Feature Representations

Frozen ResNet-50, ViT-B/16, and OpenCLIP ViT-B-32 with controlled STL-10 interventions.

**Execution policy:** this notebook is intentionally delivered unexecuted. Set the configuration paths and switches, then run top-to-bottom when you are ready to conduct the experiment. It does not answer the report questions.

## 1. Configuration

The defaults implement the prescribed STL-10 protocol. Choose the additional color intervention and cue-conflict pairs before running, then record the hypotheses in your report.

The next cell defines all reproducibility-sensitive choices: the fixed seed, dataset location, held-out subset size, color intervention, conflict pairs, and display switches. It produces no results.

In Colab, this configuration mounts Drive. Large inputs and checkpoints live under `ATML_PA1/` in Drive; small final outputs go to this task's repository `results/` directory. The repository name is detected automatically.

In [ ]:
# Standard library and experiment dependencies.
# Install dependencies yourself before executing: torch torchvision open_clip_torch
# scikit-learn pandas matplotlib seaborn pillow scipy tqdm
import json, random, math, copy
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageOps
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, roc_auc_score, roc_curve
from sklearn.linear_model import LogisticRegression
from sklearn.manifold import TSNE
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets, models, transforms
from torchvision.transforms import InterpolationMode

SEED = 6304
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


### Repository and Drive paths

The next cell locates the clone, mounts Drive in Colab, and creates persistent data/checkpoint/artifact directories plus the task's small-results directory.

In [ ]:
def find_project_root():
    """Find the cloned checkout without assuming its directory name."""
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "requirements.txt").is_file() and (
            candidate / "task1" / "task1.ipynb"
        ).is_file():
            return candidate
    # Colab often starts in /content even when the notebook lives in a cloned repo.
    content_root = Path("/content")
    if content_root.is_dir():
        matches = [
            child
            for child in content_root.iterdir()
            if child.is_dir()
            and (child / "requirements.txt").is_file()
            and (child / "task1" / "task1.ipynb").is_file()
        ]
        if len(matches) == 1:
            return matches[0]
    raise FileNotFoundError(
        "Change into the cloned repository (or a task folder); exactly one PA1 clone must be discoverable under /content."
    )


# The clone holds code and small final outputs; Drive retains large inputs and models.
REPO_ROOT = find_project_root()
TASK_ROOT = REPO_ROOT / "task1"
REPO_RESULTS_DIR = TASK_ROOT / "results"

import os

if "COLAB_RELEASE_TAG" in os.environ:
    from google.colab import drive

    drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/ATML_PA1")
DATA_ROOT = DRIVE_ROOT / "data"
CHECKPOINT_DIR = DRIVE_ROOT / "checkpoints" / "task1"
ARTIFACT_DIR = DRIVE_ROOT / "artifacts" / "task1"
EXTERNAL_DIR = DRIVE_ROOT / "external"
TORCH_CACHE_DIR = EXTERNAL_DIR / "torch_cache"
HF_CACHE_DIR = EXTERNAL_DIR / "huggingface_cache"

for directory in (
    REPO_RESULTS_DIR,
    DATA_ROOT,
    CHECKPOINT_DIR,
    ARTIFACT_DIR,
    EXTERNAL_DIR,
    TORCH_CACHE_DIR,
    HF_CACHE_DIR,
):
    directory.mkdir(parents=True, exist_ok=True)

# Pretrained-weight downloads also survive a Colab runtime reset.
os.environ["TORCH_HOME"] = str(TORCH_CACHE_DIR)
os.environ["HF_HOME"] = str(HF_CACHE_DIR)


### Reproducibility and display helpers

The next cell defines the fixed-seed behavior and small output helpers. It does not run an experiment.

In [ ]:
# Reusing one seed makes the split, subset, permutations, and head comparison reproducible.
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def save_json(value, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(value, f, indent=2)


def show_table(rows, title=None):
    frame = pd.DataFrame(rows)
    if title:
        print(title)
    display(frame)
    return frame


set_seed()


### Task settings

The next cell records this task's data path, fixed experiment settings, and plot/print_metrics switches. It produces no metrics.

In [ ]:
import open_clip

CONFIG = {
    "data_root": DATA_ROOT,
    "download_data": False,
    "results_dir": REPO_RESULTS_DIR,
    "batch_size": 64,
    "head_epochs": 50,
    "head_patience": 5,
    "learning_rate": 1e-3,
    "weight_decay": 1e-4,
    "test_subset_per_class": 50,
    "additional_color": "hue_rotation_90",
    "cue_pairs": [
        ("airplane", "bird"),
        ("car", "truck"),
        ("cat", "dog"),
        ("deer", "horse"),
        ("monkey", "ship"),
    ],
    "adain_alpha": 0.8,
    "visual_rejection_rule": "Reject if content object is not recognizable or style texture is not visibly transferred; never use predictions.",
    "plot": True,
    "print_metrics": True,
}
RESULTS = Path(CONFIG["results_dir"])
RESULTS.mkdir(parents=True, exist_ok=True)
CLASSES = ["airplane", "bird", "car", "cat", "deer", "dog", "horse", "monkey", "ship", "truck"]


## 2. Data split and shared 224x224 interventions

All backbones receive identical RGB images before their model-specific normalization. The saved manifest makes the class-balanced test subset recoverable.

The next cell creates the fixed stratified 80/20 train/validation split and 500-image balanced test manifest, then defines shared pixel-space interventions. It saves `test_subset_manifest.json`.

In [ ]:
# Keep the existing torchvision loader and its explicit download switch.
if not CONFIG["download_data"] and not (DATA_ROOT / "stl10_binary").is_dir():
    raise FileNotFoundError(f"STL-10 is missing from Google Drive: {DATA_ROOT / 'stl10_binary'}")
raw_train = datasets.STL10(CONFIG["data_root"], split="train", download=CONFIG["download_data"])
raw_test = datasets.STL10(CONFIG["data_root"], split="test", download=CONFIG["download_data"])
labels = np.asarray(raw_train.labels)
rng = np.random.default_rng(SEED)
train_indices, val_indices = [], []
for label in range(10):
    indices = np.flatnonzero(labels == label)
    rng.shuffle(indices)
    cut = int(0.8 * len(indices))
    train_indices.extend(indices[:cut])
    val_indices.extend(indices[cut:])
test_labels = np.asarray(raw_test.labels)
chosen_test = []
for label in range(10):
    candidates = np.flatnonzero(test_labels == label)
    rng.shuffle(candidates)
    chosen_test.extend(candidates[: CONFIG["test_subset_per_class"]])
save_json(
    {"seed": SEED, "indices": list(map(int, chosen_test))}, RESULTS / "test_subset_manifest.json"
)


### Shared image interventions

The following functions define the common image view and deterministic interventions. They do not train or evaluate a model.

In [ ]:
def common_image(image):
    # Interventions operate on one shared RGB/crop view before backbone normalization.
    rgb_image = image.convert("RGB")
    resized_image = transforms.Resize(256, InterpolationMode.BICUBIC)(rgb_image)
    return transforms.CenterCrop(224)(resized_image)


def grayscale(image):
    return ImageOps.grayscale(image).convert("RGB")


def hue_rotation_90(image):
    hsv = np.asarray(image.convert("HSV")).copy()
    hsv[..., 0] = (hsv[..., 0].astype(np.uint16) + 64) % 256
    return Image.fromarray(hsv, "HSV").convert("RGB")


def translate_reflect(image, pixels, direction):
    array = np.asarray(image)
    pad = pixels
    padded = np.pad(array, ((pad, pad), (pad, pad), (0, 0)), mode="reflect")
    dx, dy = {"left": (-pixels, 0), "right": (pixels, 0), "up": (0, -pixels), "down": (0, pixels)}[
        direction
    ]
    y, x = pad + dy, pad + dx
    return Image.fromarray(padded[y : y + array.shape[0], x : x + array.shape[1]])


# The permutation is seeded from the image identifier so every backbone sees the same shuffled image.
def shuffle_4x4(image, index):
    array = np.asarray(image)
    h, w = array.shape[:2]
    rng = np.random.default_rng(SEED + int(index))
    order = rng.permutation(16)
    # A non-identity shuffle is required even if the random draw returns the original order.
    if np.array_equal(order, np.arange(16)):
        order = np.roll(order, 1)

    pieces = []
    for row in range(4):
        patch_row = []
        for column in range(4):
            patch = array[
                row * h // 4 : (row + 1) * h // 4,
                column * w // 4 : (column + 1) * w // 4,
            ]
            patch_row.append(patch)
        pieces.append(patch_row)
    out = np.empty_like(array)
    for target, source in enumerate(order):
        ti, tj = divmod(target, 4)
        si, sj = divmod(source, 4)
        out[ti * h // 4 : (ti + 1) * h // 4, tj * w // 4 : (tj + 1) * w // 4] = pieces[si][sj]
    return Image.fromarray(out)


## 3. Frozen representations and linear heads

The next cell loads the three required pretrained backbones, freezes them, extracts final representations, and trains one early-stopped linear head per backbone. Its outputs are trained heads and feature tensors.

In [ ]:
resnet_w = models.ResNet50_Weights.IMAGENET1K_V2
vit_w = models.ViT_B_16_Weights.IMAGENET1K_V1
resnet = models.resnet50(weights=resnet_w)
resnet.fc = nn.Identity()
vit = models.vit_b_16(weights=vit_w)
vit.heads = nn.Identity()
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32", pretrained="openai"
)
for model in (resnet, vit, clip_model):
    model.to(DEVICE).eval()
    for parameter in model.parameters():
        parameter.requires_grad = False
# The shared 224x224 image is already cropped. Apply only each model's normalization here;
# weight presets otherwise resize/crop it again and change the intervention itself.
clip_normalize = next(
    step for step in clip_preprocess.transforms if isinstance(step, transforms.Normalize)
)
normalizers = {
    "resnet": transforms.Compose(
        [
            transforms.ToTensor(),
            transforms.Normalize(resnet_w.transforms().mean, resnet_w.transforms().std),
        ]
    ),
    "vit": transforms.Compose(
        [
            transforms.ToTensor(),
            transforms.Normalize(vit_w.transforms().mean, vit_w.transforms().std),
        ]
    ),
    "clip": transforms.Compose([transforms.ToTensor(), clip_normalize]),
}
backbones = {"resnet": resnet, "vit": vit, "clip": clip_model}


# Each model receives the same PIL image; only this step applies its required normalization.
def feature(model_name, image_batch):
    # image_batch is a list of common PIL images; normalization happens only here.
    tensor = torch.stack([normalizers[model_name](im) for im in image_batch]).to(DEVICE)
    with torch.inference_mode():
        if model_name == "clip":
            output = clip_model.encode_image(tensor)
        else:
            output = backbones[model_name](tensor)
        # CLIP specifies unit-normalized image embeddings. Keep raw pooled/token features for the other two.
        output = output.float()
        if model_name == "clip":
            output = F.normalize(output, dim=1)
        return output.cpu()


def extract(model_name, dataset, indices, transform=lambda im, image_id: im):
    features, ys = [], []
    for start in range(0, len(indices), CONFIG["batch_size"]):
        batch_ids = indices[start : start + CONFIG["batch_size"]]
        samples = [dataset[int(i)] for i in batch_ids]
        images = [
            transform(common_image(sample[0]), int(image_id))
            for sample, image_id in zip(samples, batch_ids)
        ]
        features.append(feature(model_name, images))
        ys.extend(int(sample[1]) for sample in samples)
    return torch.cat(features), torch.tensor(ys)


### Fit the linear heads

The frozen backbones provide training and validation features. This cell fits only the linear heads and retains each validation-selected head in memory.

In [ ]:
train_features = {}
val_features = {}
for backbone_name in backbones:
    # Extract once: the frozen features are reused for every linear-head epoch.
    train_features[backbone_name] = extract(backbone_name, raw_train, train_indices)
    val_features[backbone_name] = extract(backbone_name, raw_train, val_indices)
heads = {}
for name, (x_train, y_train) in train_features.items():
    head = nn.Linear(x_train.shape[1], 10).to(DEVICE)
    optimizer = torch.optim.AdamW(
        head.parameters(), lr=CONFIG["learning_rate"], weight_decay=CONFIG["weight_decay"]
    )
    best, stale, best_state = -1, 0, None
    for epoch in range(CONFIG["head_epochs"]):
        head.train()
        optimizer.zero_grad()
        loss = F.cross_entropy(head(x_train.to(DEVICE)), y_train.to(DEVICE))
        loss.backward()
        optimizer.step()
        head.eval()
        score = accuracy_score(
            val_features[name][1], head(val_features[name][0].to(DEVICE)).argmax(1).cpu()
        )
        if score > best:
            best, stale, best_state = score, 0, copy.deepcopy(head.state_dict())
        else:
            stale += 1
        if stale >= CONFIG["head_patience"]:
            break
    head.load_state_dict(best_state)
    heads[name] = head.eval()


## 4. Clean, color, and patch evaluation

Set `print_metrics` or `plot` in the configuration to control notebook output.

The next cell evaluates clean, grayscale, hue-rotation, and patch-shuffle inputs. It produces top-1 accuracy, macro-F1, mean maximum confidence, accuracy change, and prediction consistency.

In [ ]:
tokenizer = open_clip.get_tokenizer("ViT-B-32")
with torch.inference_mode():
    text = F.normalize(
        clip_model.encode_text(
            tokenizer([f"a photo of a {name}." for name in CLASSES]).to(DEVICE)
        ).float(),
        dim=1,
    )


def predict(name, feats):
    if name == "clip_zero_shot":
        scale = clip_model.logit_scale.exp()
        return (scale * feats.to(DEVICE) @ text.T).softmax(1).cpu()
    return heads[name](feats.to(DEVICE)).softmax(1).cpu()


def evaluate_condition(name, transform):
    backbone_name = "clip" if name == "clip_zero_shot" else name
    feats, y = extract(backbone_name, raw_test, chosen_test, transform)
    probs = predict(name, feats)
    pred = probs.argmax(1)
    return {
        "accuracy": accuracy_score(y, pred),
        "macro_f1": f1_score(y, pred, average="macro"),
        "mean_max_confidence": probs.max(1).values.mean().item(),
        "pred": pred,
        "features": feats,
        "labels": y,
    }


### Clean, color, and patch measurements

This cell evaluates the fixed test subset, displays the comparison table, and writes machine-readable condition metrics. The intervention definitions above remain unchanged.

In [ ]:
conditions = {
    "clean": lambda image, image_id: image,
    "grayscale": lambda image, image_id: grayscale(image),
    "hue_rotation_90": lambda image, image_id: hue_rotation_90(image),
    "patch_shuffle": None,
}
all_results = {}
for model_name in ["resnet", "vit", "clip", "clip_zero_shot"]:
    clean = evaluate_condition(model_name, conditions["clean"])
    all_results[(model_name, "clean")] = clean
    for condition in ["grayscale", "hue_rotation_90"]:
        result = evaluate_condition(model_name, conditions[condition])
        result["consistency"] = (result["pred"] == clean["pred"]).float().mean().item()
        all_results[(model_name, condition)] = result
    # Each test image uses a distinct but deterministic non-identity permutation.
    shuffled = extract(
        "clip" if model_name == "clip_zero_shot" else model_name,
        raw_test,
        chosen_test,
        lambda image, image_id: shuffle_4x4(image, image_id),
    )
    y = shuffled[1]
    probs = predict(model_name, shuffled[0])
    pred = probs.argmax(1)
    all_results[(model_name, "patch_shuffle")] = {
        "accuracy": accuracy_score(y, pred),
        "macro_f1": f1_score(y, pred, average="macro"),
        "mean_max_confidence": probs.max(1).values.mean().item(),
        "pred": pred,
        "features": shuffled[0],
        "labels": y,
        "consistency": (pred == clean["pred"]).float().mean().item(),
    }
rows = []
for (model, condition), value in all_results.items():
    row = {
        "model": model,
        "condition": condition,
        "accuracy": round(value["accuracy"], 4),
        "macro_f1": round(value["macro_f1"], 4),
        "mean_max_confidence": round(value["mean_max_confidence"], 4),
    }
    if condition != "clean":
        row["accuracy_change"] = round(
            value["accuracy"] - all_results[(model, "clean")]["accuracy"], 4
        )
        row["consistency"] = round(value["consistency"], 4)
    rows.append(row)
if CONFIG["print_metrics"]:
    show_table(rows, "Clean and intervention metrics")
save_json(rows, RESULTS / "condition_metrics.json")


## 5. Translation curve

The next cell evaluates the four cardinal translations at 0, 8, 16, and 32 pixels. It saves a metric table and plots accuracy and prediction consistency against displacement.

In [ ]:
translation_rows = []
for model_name in ["resnet", "vit", "clip", "clip_zero_shot"]:
    clean_pred = all_results[(model_name, "clean")]["pred"]
    for pixels in [0, 8, 16, 32]:
        direction_predictions = []
        for direction in ["left", "right", "up", "down"]:
            transformed = evaluate_condition(
                model_name, lambda im, image_id, p=pixels, d=direction: translate_reflect(im, p, d)
            )
            direction_predictions.append(transformed["pred"])
        stacked = torch.stack(direction_predictions)
        y = all_results[(model_name, "clean")]["labels"]
        translation_rows.append(
            {
                "model": model_name,
                "pixels": pixels,
                "accuracy": np.mean([accuracy_score(y, p) for p in direction_predictions]),
                "consistency": (stacked == clean_pred).float().mean().item(),
            }
        )
translation_frame = pd.DataFrame(translation_rows)
save_json(translation_rows, RESULTS / "translation_metrics.json")
if CONFIG["print_metrics"]:
    print("Translation: mean over four directions at each displacement")
    display(translation_frame.round(4))
if CONFIG["plot"]:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for model, frame in translation_frame.groupby("model"):
        axes[0].plot(frame.pixels, frame.accuracy, "o-", label=model)
        axes[1].plot(frame.pixels, frame.consistency, "o-", label=model)
    axes[0].set(title="Translation accuracy", xlabel="displacement (pixels)", ylabel="accuracy")
    axes[1].set(
        title="Prediction consistency", xlabel="displacement (pixels)", ylabel="consistency"
    )
    axes[1].legend()
    fig.savefig(REPO_RESULTS_DIR / "translation_accuracy_consistency.png", dpi=150, bbox_inches="tight")
    plt.show()


## 6. Cue conflicts (AdaIN)

Install the cited AdaIN implementation and point `ADAIN_DIR` to it before executing this section. Generate conflicts, inspect them using the rejection rule configured above, record accepted/rejected IDs, and only then evaluate.

The following setup creates an auditable conflict manifest before any prediction. After manual visual acceptance, it computes shape, texture, other, shape-bias, and coverage counts.

In [ ]:
# AdaIN is used only to generate images. Predictions are deliberately not available in this section,
# so visual acceptance cannot be influenced by any model output.
ADAIN_DIR = EXTERNAL_DIR / "pytorch-AdaIN"
ADAIN_DECODER = ADAIN_DIR / "models" / "decoder.pth"
ADAIN_VGG = ADAIN_DIR / "models" / "vgg_normalised.pth"
CONFLICT_DIR = ARTIFACT_DIR / "cue_conflicts"
CONFLICT_MANIFEST = RESULTS / "cue_conflict_manifest.json"


def load_adain_generator():
    """Load the separately installed, cited AdaIN implementation and its published weights."""
    if not ADAIN_DIR.exists():
        raise FileNotFoundError(
            f"Install the cited pytorch-AdaIN repository in Google Drive at {ADAIN_DIR}."
        )
    if not ADAIN_DECODER.exists() or not ADAIN_VGG.exists():
        raise FileNotFoundError(
            f"Place decoder.pth and vgg_normalised.pth in Google Drive at {ADAIN_DIR / 'models'}."
        )

    import sys

    if str(ADAIN_DIR) not in sys.path:
        sys.path.insert(0, str(ADAIN_DIR))
    import net as adain_net

    encoder = adain_net.vgg
    decoder = adain_net.decoder
    encoder.load_state_dict(torch.load(ADAIN_VGG, map_location=DEVICE))
    decoder.load_state_dict(torch.load(ADAIN_DECODER, map_location=DEVICE))
    # The AdaIN reference uses only the encoder portion needed for content/style statistics.
    encoder = nn.Sequential(*list(encoder.children())[:31])
    # The reference inference path uses VGG features, AdaIN, and the decoder directly.
    return encoder.to(DEVICE).eval(), decoder.to(DEVICE).eval()


def adain_stylize(generator, content_image, style_image, alpha):
    """Return one 224x224 RGB conflict image; alpha controls the content/style trade-off."""
    to_tensor = transforms.ToTensor()
    content = to_tensor(content_image).unsqueeze(0).to(DEVICE)
    style = to_tensor(style_image).unsqueeze(0).to(DEVICE)
    with torch.inference_mode():
        from function import adaptive_instance_normalization

        encoder, decoder = generator
        content_features = encoder(content)
        style_features = encoder(style)
        stylized_features = adaptive_instance_normalization(content_features, style_features)
        blended_features = alpha * stylized_features + (1 - alpha) * content_features
        output = decoder(blended_features).squeeze(0).clamp(0, 1).cpu()
    return transforms.ToPILImage()(output).convert("RGB")


def build_conflict_candidates():
    """Create both directions for every chosen unordered pair before any visual review."""
    candidates = []
    for shape_name, texture_name in CONFIG["cue_pairs"]:
        first_label = CLASSES.index(shape_name)
        second_label = CLASSES.index(texture_name)
        for shape_label, texture_label in [
            (first_label, second_label),
            (second_label, first_label),
        ]:
            content_ids = [index for index in chosen_test if raw_test.labels[index] == shape_label]
            style_ids = [index for index in chosen_test if raw_test.labels[index] == texture_label]
            for content_id, style_id in zip(content_ids, style_ids):
                candidates.append(
                    {
                        "content_id": int(content_id),
                        "style_id": int(style_id),
                        "shape_label": int(shape_label),
                        "texture_label": int(texture_label),
                        "image_file": f"conflict_{content_id}_{style_id}.png",
                        "accepted": None,
                        "rejection_reason": None,
                    }
                )
    return candidates


### Generate and visually review cue-conflict images

The following functions create AdaIN candidates in Drive and display review pages. The small acceptance manifest is in the repository. Review decisions must be recorded before any model prediction is used.

In [ ]:
def generate_conflicts():
    """Generate candidates once, save an auditable manifest, and return records for visual review."""
    # Preserve the human visual-review decisions if this cell is revisited.
    if CONFLICT_MANIFEST.exists():
        records = json.load(open(CONFLICT_MANIFEST))
        missing_images = [
            record["image_file"]
            for record in records
            if not (CONFLICT_DIR / record["image_file"]).is_file()
        ]
        if missing_images:
            raise FileNotFoundError(
                f"Reviewed cue-conflict images are missing from Google Drive: {CONFLICT_DIR / missing_images[0]}"
            )
        return records
    generator = load_adain_generator()
    records = build_conflict_candidates()
    CONFLICT_DIR.mkdir(parents=True, exist_ok=True)
    for record in records:
        output_path = CONFLICT_DIR / record["image_file"]
        if not output_path.exists():
            content = common_image(raw_test[record["content_id"]][0])
            style = common_image(raw_test[record["style_id"]][0])
            adain_stylize(generator, content, style, CONFIG["adain_alpha"]).save(output_path)
    save_json(records, CONFLICT_MANIFEST)
    return records


def show_conflict_candidates(start=0, count=12):
    """Display a page of stylizations for the visual review, before model evaluation."""
    if not CONFIG["plot"]:
        return
    records = json.load(open(CONFLICT_MANIFEST))
    page = records[start : start + count]
    columns = 4
    rows = math.ceil(len(page) / columns)
    figure, axes = plt.subplots(rows, columns, figsize=(12, 3 * rows))
    for axis, record in zip(np.ravel(axes), page):
        image = Image.open(CONFLICT_DIR / record["image_file"]).convert("RGB")
        axis.imshow(image)
        shape_name = CLASSES[record["shape_label"]]
        texture_name = CLASSES[record["texture_label"]]
        axis.set_title(
            f"shape: {shape_name} / texture: {texture_name}\n" f"{record['image_file']}",
            fontsize=8,
        )
        axis.axis("off")
    for axis in list(np.ravel(axes))[len(page) :]:
        axis.axis("off")
    figure.tight_layout()
    plt.show()


# Run `generate_conflicts`, inspect the saved images without model predictions, and edit ONLY
# accepted/rejection_reason in the manifest. The configured rejection rule is recorded in CONFIG.
# Do not evaluate until at least 200 visually valid conflicts have accepted=True.
def accepted_conflict_records():
    records = json.load(open(CONFLICT_MANIFEST))
    accepted = [record for record in records if record["accepted"] is True]
    rejected = [record for record in records if record["accepted"] is False]
    if len(accepted) < 200:
        raise ValueError(
            "At least 200 visually accepted cue conflicts are required before evaluation."
        )
    save_json(
        {
            "accepted": len(accepted),
            "rejected": len(rejected),
            "rule": CONFIG["visual_rejection_rule"],
        },
        RESULTS / "cue_conflict_acceptance_counts.json",
    )
    return accepted


def conflict_features(backbone_name, records):
    """Extract each model's features from exactly the same accepted conflict image files."""
    feature_batches = []
    for start in range(0, len(records), CONFIG["batch_size"]):
        batch = records[start : start + CONFIG["batch_size"]]
        images = [
            Image.open(CONFLICT_DIR / record["image_file"]).convert("RGB") for record in batch
        ]
        feature_batches.append(feature(backbone_name, images))
    return torch.cat(feature_batches)


### Cue-conflict evaluation

After at least 200 images pass visual review, the following functions produce shape/texture/other summaries, a per-image CSV, and a final example figure.

In [ ]:
def cue_summary(predictions, records):
    shape_count = sum(
        int(prediction) == record["shape_label"] for prediction, record in zip(predictions, records)
    )
    texture_count = sum(
        int(prediction) == record["texture_label"]
        for prediction, record in zip(predictions, records)
    )
    total = len(records)
    eligible = shape_count + texture_count
    return {
        "shape": shape_count,
        "texture": texture_count,
        "other": total - eligible,
        "shape_bias_percent": 100 * shape_count / max(eligible, 1),
        "coverage_percent": 100 * eligible / max(total, 1),
    }


def evaluate_cue_conflicts():
    """Classify accepted conflicts as shape, texture, or other for every required model."""
    records = accepted_conflict_records()
    rows = []
    prediction_rows = []
    for model_name in ["resnet", "vit", "clip", "clip_zero_shot"]:
        backbone_name = "clip" if model_name == "clip_zero_shot" else model_name
        probabilities = predict(model_name, conflict_features(backbone_name, records))
        predictions = probabilities.argmax(1)
        rows.append({"model": model_name, **cue_summary(predictions, records)})
        for record, prediction in zip(records, predictions):
            predicted_index = int(prediction)
            if predicted_index == record["shape_label"]:
                decision = "shape"
            elif predicted_index == record["texture_label"]:
                decision = "texture"
            else:
                decision = "other"
            prediction_rows.append(
                {
                    "image_file": record["image_file"],
                    "content_id": record["content_id"],
                    "style_id": record["style_id"],
                    "shape_class": CLASSES[record["shape_label"]],
                    "texture_class": CLASSES[record["texture_label"]],
                    "model": model_name,
                    "predicted_class": CLASSES[predicted_index],
                    "decision": decision,
                }
            )
    save_json(rows, RESULTS / "cue_conflict_metrics.json")
    # The per-image table lets the report select agreements, disagreements, and failures.
    prediction_frame = pd.DataFrame(prediction_rows)
    prediction_frame.to_csv(RESULTS / "cue_conflict_predictions.csv", index=False)
    if CONFIG["print_metrics"]:
        show_table(rows, "Cue-conflict decisions")
        # A few decisions of each type make the saved per-image table easy to inspect.
        examples = []
        for decision in ["shape", "texture", "other"]:
            examples.append(prediction_frame[prediction_frame["decision"] == decision].head(2))
        print(f"Example cue-conflict decisions (image files are in {CONFLICT_DIR})")
        display(pd.concat(examples, ignore_index=True))
    if CONFIG["plot"]:
        # Prefer images on which the four models disagree; fill remaining slots in file order.
        predictions_by_image = prediction_frame.pivot(
            index="image_file", columns="model", values="predicted_class"
        )
        differing = predictions_by_image.nunique(axis=1) > 1
        chosen_files = list(predictions_by_image.index[differing][:6])
        for image_file in predictions_by_image.index:
            if len(chosen_files) == 6:
                break
            if image_file not in chosen_files:
                chosen_files.append(image_file)
        record_by_file = {record["image_file"]: record for record in records}
        figure, axes = plt.subplots(2, 3, figsize=(14, 9))
        for axis, image_file in zip(axes.flat, chosen_files):
            record = record_by_file[image_file]
            axis.imshow(Image.open(CONFLICT_DIR / image_file).convert("RGB"))
            decisions = predictions_by_image.loc[image_file]
            caption = (
                f"shape {CLASSES[record['shape_label']]} / "
                f"texture {CLASSES[record['texture_label']]}\n"
                f"R:{decisions['resnet']} V:{decisions['vit']} "
                f"C:{decisions['clip']} Z:{decisions['clip_zero_shot']}"
            )
            axis.set_title(caption, fontsize=9)
            axis.axis("off")
        for axis in list(axes.flat)[len(chosen_files) :]:
            axis.axis("off")
        figure.tight_layout()
        figure.savefig(REPO_RESULTS_DIR / "cue_conflict_examples.png", dpi=150, bbox_inches="tight")
        plt.show()
    return rows


## 7. Representation stability and projections

The next cell compares paired clean/transformed backbone features. It produces cosine-stability rows and one combined-condition t-SNE plot per backbone/intervention.

In [ ]:
# Fit each projection on a clean/transformed pair from the same images. Coordinates are therefore
# comparable only within one backbone/intervention panel, exactly as the manual specifies.
def paired_representation_features(backbone_name, intervention):
    if intervention == "grayscale":
        return (
            all_results[(backbone_name, "clean")]["features"],
            all_results[(backbone_name, "grayscale")]["features"],
            all_results[(backbone_name, "clean")]["labels"],
        )
    if intervention == "patch_shuffle":
        return (
            all_results[(backbone_name, "clean")]["features"],
            all_results[(backbone_name, "patch_shuffle")]["features"],
            all_results[(backbone_name, "clean")]["labels"],
        )
    if intervention == "translation_32_right":
        clean_features, labels = extract(backbone_name, raw_test, chosen_test)
        translated_features, _ = extract(
            backbone_name,
            raw_test,
            chosen_test,
            lambda image, image_id: translate_reflect(image, 32, "right"),
        )
        return clean_features, translated_features, labels
    if intervention == "cue_conflict":
        records = accepted_conflict_records()
        content_ids = [record["content_id"] for record in records]
        clean_features, labels = extract(backbone_name, raw_test, content_ids)
        conflict_features_for_model = conflict_features(backbone_name, records)
        return clean_features, conflict_features_for_model, labels
    raise ValueError(f"Unknown intervention: {intervention}")


### Representation measurements

The next cell evaluates paired clean/transformed features for each backbone, displays t-SNE panels when plotting is enabled, and saves cosine-stability values and final panels.

In [ ]:
stability_rows = []
for backbone_name in ["resnet", "vit", "clip"]:
    for intervention in ["grayscale", "cue_conflict", "translation_32_right", "patch_shuffle"]:
        clean_features, transformed_features, labels = paired_representation_features(
            backbone_name, intervention
        )
        cosine_stability = F.cosine_similarity(clean_features, transformed_features).mean().item()
        stability_rows.append(
            {
                "backbone": backbone_name,
                "intervention": intervention,
                "cosine_stability": cosine_stability,
            }
        )

        combined_features = torch.cat([clean_features, transformed_features]).numpy()
        projection = TSNE(
            n_components=2, random_state=SEED, init="pca", learning_rate="auto", perplexity=30
        ).fit_transform(combined_features)
        condition = np.array(
            ["clean"] * len(clean_features) + [intervention] * len(transformed_features)
        )
        plot_data = pd.DataFrame(
            {
                "x": projection[:, 0],
                "y": projection[:, 1],
                "class": np.tile(labels.numpy(), 2),
                "condition": condition,
            }
        )
        if CONFIG["plot"]:
            sns.scatterplot(
                data=plot_data, x="x", y="y", hue="class", style="condition", palette="tab10", s=25
            )
            plt.title(f"{backbone_name}: clean vs {intervention}")
            plt.gcf().savefig(
                REPO_RESULTS_DIR / f"representation_{backbone_name}_{intervention}.png",
                dpi=150,
                bbox_inches="tight",
            )
            plt.show()

if CONFIG["print_metrics"]:
    show_table(stability_rows, "Representation cosine stability")
save_json(stability_rows, RESULTS / "representation_stability.json")
